# Olist SQL — Week 2 Complete Answer Key

> **Database:** Olist Brazilian E-Commerce Dataset  
> **Database engine:** SQLite  
> **Difficulty:**  Intermediate Relational Concepts  
> **Questions:** 1–25

---

# Part I — INNER JOIN

## 1. Customer Orders

### Question

Return:

- `customer_id`
- `customer_unique_id`
- `order_id`

for every customer who has placed an order.

### Answer

```sql
SELECT
    c.customer_id,
    c.customer_unique_id,
    o.order_id
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id;
```

### Explanation

The relationship is:

```
customers.customer_id
        ↓
orders.customer_id
```

`INNER JOIN` returns only customers who have a matching order.

If a customer has never placed an order, that customer does not appear.

---

# Part II — WHERE + INNER JOIN

## 2. Delivered Orders

### Question

Return:

- `order_id`
- `customer_id`
- `order_status`

for orders that have been delivered.

### Answer

```sql
SELECT
    o.order_id,
    o.customer_id,
    o.order_status
FROM orders AS o
WHERE o.order_status = 'delivered';
```

### Explanation

`WHERE` filters individual rows.

Only rows where:

```sql
order_status = 'delivered'
```

remain.

---

# Part III — JOIN + WHERE

## 3. Customers With Delivered Orders

### Question

Return:

- `customer_unique_id`
- `order_id`

for customers who have delivered orders.

### Answer

```sql
SELECT
    c.customer_unique_id,
    o.order_id
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered';
```

### Explanation

The `JOIN` connects the customer to the order.

The `WHERE` then keeps only delivered orders.

Logical idea:

```
customers
    ↓
JOIN orders
    ↓
filter delivered orders
```

---

# Part IV — LEFT JOIN

## 4. Every Customer and Their Orders

### Question

Return:

- `customer_id`
- `customer_unique_id`
- `order_id`

Include customers who have never placed an order.

### Answer

```sql
SELECT
    c.customer_id,
    c.customer_unique_id,
    o.order_id
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id;
```

### Explanation

The important word is:

> every customer

Therefore, customers must be the left table.

```
customers
    LEFT JOIN
orders
```

A customer without an order receives:

```sql
order_id = NULL
```

This is the fundamental purpose of a `LEFT JOIN`.

---

# Part V — LEFT JOIN + NULL

## 5. Customers With No Orders

### Question

Find customers who have never placed an order.

### Answer

```sql
SELECT
    c.customer_id,
    c.customer_unique_id
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;
```

### Explanation

The `LEFT JOIN` preserves every customer.

Customers with no matching order receive `NULL` in the order columns.

Therefore:

```sql
WHERE o.order_id IS NULL
```

finds customers with no orders.

Mental model:

```
LEFT JOIN
    ↓
preserve everybody on the left
    ↓
NULL on the right = no match
```

---

# Part VI — RIGHT JOIN

## 6. Rewrite a LEFT JOIN as a RIGHT JOIN

### Question

Produce the equivalent result of Question 4 using `RIGHT JOIN`.

### Answer

```sql
SELECT
    c.customer_id,
    c.customer_unique_id,
    o.order_id
FROM orders AS o
RIGHT JOIN customers AS c
    ON o.customer_id = c.customer_id;
```

### Explanation

These two queries are logically equivalent:

```sql
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
```

and:

```sql
FROM orders AS o
RIGHT JOIN customers AS c
    ON o.customer_id = c.customer_id
```

The difference is simply which table is considered the preserved side.

**Important**

In practical SQL, `LEFT JOIN` is generally easier to read because you can consistently put the table you want to preserve on the left.

SQLite supports `RIGHT JOIN` and `FULL OUTER JOIN` in current versions.

---

# Part VII — FULL OUTER JOIN

## 7. Customers and Orders With FULL OUTER JOIN

### Question

Use a `FULL OUTER JOIN` between customers and orders.

Return:

- `customer_id`
- `order_id`

### Answer

```sql
SELECT
    c.customer_id,
    o.order_id
FROM customers AS c
FULL OUTER JOIN orders AS o
    ON c.customer_id = o.customer_id;
```

### Explanation

A `FULL OUTER JOIN` preserves:

- matching rows
- unmatched left rows
- unmatched right rows

Conceptually:

```
customers       orders
    │              │
    └──── JOIN ────┘
          │
    everything
```

Unmatched values become `NULL`.

---

# Part VIII — NATURAL JOIN

## 8. Explore NATURAL JOIN

### Question

Use `NATURAL JOIN` to connect appropriate Olist tables.

### Answer

A simple example is:

```sql
SELECT
    *
FROM orders
NATURAL JOIN order_items;
```

### Explanation

A `NATURAL JOIN` automatically joins tables using columns that have the same names.

This can be convenient, but it is dangerous in production SQL.

You normally want to explicitly state:

```sql
ON o.order_id = oi.order_id
```

rather than allowing the database to decide which identically named columns should be used.

**Better engineering practice**

Prefer:

```sql
SELECT
    *
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id;
```

instead of:

```sql
SELECT *
FROM orders
NATURAL JOIN order_items;
```

**Why?**

If a table later gains another column with the same name, the behavior of the `NATURAL JOIN` can unexpectedly change.

For learning purposes, understand `NATURAL JOIN`.

For professional production SQL, prefer explicit join conditions.

---

# Part IX — GROUP BY

## 9. Number of Orders Per Customer

### Question

Find the number of orders placed by each customer.

Return:

- `customer_id`
- order count

### Answer

```sql
SELECT
    customer_id,
    COUNT(*) AS order_count
FROM orders
GROUP BY customer_id;
```

### Explanation

`GROUP BY` creates one group for each customer.

Example:

```
customer 1
├── order A
├── order B
└── order C

customer 2
├── order D
└── order E
```

Then:

```sql
COUNT(*)
```

counts the rows inside each group.

---

# Part X — GROUP BY + ORDER BY

## 10. Top Customers by Number of Orders

### Question

Return the top 10 customers based on number of orders.

### Answer

```sql
SELECT
    customer_id,
    COUNT(*) AS order_count
FROM orders
GROUP BY customer_id
ORDER BY order_count DESC
LIMIT 10;
```

### Explanation

The query:

1. Groups orders by customer.
2. Counts each customer's orders.
3. Sorts from largest to smallest.
4. Returns the first 10.

---

# Part XI — HAVING

## 11. Customers With More Than 5 Orders

### Question

Find customers who have placed more than 5 orders.

### Answer

```sql
SELECT
    customer_id,
    COUNT(*) AS order_count
FROM orders
GROUP BY customer_id
HAVING COUNT(*) > 5;
```

### Explanation

We cannot use:

```sql
WHERE COUNT(*) > 5
```

because `COUNT(*)` is calculated for a group.

Therefore:

```sql
HAVING COUNT(*) > 5
```

is correct.

Mental model:

```
WHERE
    ↓
filter rows

GROUP BY
    ↓
create groups

HAVING
    ↓
filter groups
```

SQLite specifically documents `HAVING` as being evaluated after grouping.

---

# Part XII — IN + Subquery

## 12. Customers Who Have Delivered Orders

### Question

Find customers who have at least one delivered order.

Use a subquery with `IN`.

### Answer

```sql
SELECT
    customer_id,
    customer_unique_id
FROM customers
WHERE customer_id IN (
    SELECT customer_id
    FROM orders
    WHERE order_status = 'delivered'
);
```

### Explanation

The inner query:

```sql
SELECT customer_id
FROM orders
WHERE order_status = 'delivered'
```

produces a list of customer IDs.

The outer query asks:

> Is this customer's ID in that list?

That is what `IN` does.

---

# Part XIII — Non-Correlated Subquery

## 13. Products Above Average Price

### Question

Find products whose price is greater than the average product price.

### Answer

```sql
SELECT
    product_id,
    product_category_name,
    price
FROM products
WHERE price > (
    SELECT AVG(price)
    FROM products
);
```

### Explanation

The inner query:

```sql
SELECT AVG(price)
FROM products
```

does not depend on the outer query.

Therefore it is a:

> non-correlated subquery

Conceptually:

```
Calculate average
       ↓
   120.50
       ↓
Find products > 120.50
```

---

# Part XIV — EXISTS

## 14. Customers With At Least One Order

### Question

Find customers who have at least one order.

Use `EXISTS`.

### Answer

```sql
SELECT
    c.customer_id,
    c.customer_unique_id
FROM customers AS c
WHERE EXISTS (
    SELECT 1
    FROM orders AS o
    WHERE o.customer_id = c.customer_id
);
```

### Explanation

The subquery asks:

> Does an order exist for this customer?

If yes:

```
EXISTS = TRUE
```

The customer is returned.

If no:

```
EXISTS = FALSE
```

The customer is removed.

---

# Part XV — Correlated Subquery

## 15. Customers With a Delivered Order

### Question

Use a correlated subquery to find customers who have at least one delivered order.

### Answer

```sql
SELECT
    c.customer_id,
    c.customer_unique_id
FROM customers AS c
WHERE EXISTS (
    SELECT 1
    FROM orders AS o
    WHERE o.customer_id = c.customer_id
      AND o.order_status = 'delivered'
);
```

### Explanation

The important part is:

```sql
o.customer_id = c.customer_id
```

The inner query references:

```sql
c.customer_id
```

from the outer query.

Therefore the subquery depends on the current outer row.

That makes it a:

> correlated subquery

Conceptually:

```
Customer A
    ↓
Does A have a delivered order?

Customer B
    ↓
Does B have a delivered order?

Customer C
    ↓
Does C have a delivered order?
```

---

# Part XVI — IN vs EXISTS

## 16. Which One Should You Use?

### IN

Use `IN` when you naturally think:

> Is this value contained in a set of values?

Example:

```sql
SELECT
    customer_id
FROM customers
WHERE customer_id IN (
    SELECT customer_id
    FROM orders
);
```

### EXISTS

Use `EXISTS` when you naturally think:

> Does at least one matching row exist?

Example:

```sql
SELECT
    c.customer_id
FROM customers AS c
WHERE EXISTS (
    SELECT 1
    FROM orders AS o
    WHERE o.customer_id = c.customer_id
);
```

Both are useful.

Do not treat one as universally better.

---

# Part XVII — UNION

## 17. Customers From Two States

### Question

Return customer IDs for customers from São Paulo (SP) or Rio de Janeiro (RJ).

Use `UNION`.

### Answer

```sql
SELECT customer_id
FROM customers
WHERE customer_state = 'SP'

UNION

SELECT customer_id
FROM customers
WHERE customer_state = 'RJ';
```

### Explanation

`UNION` combines two result sets and removes duplicate rows.

The two queries must return compatible numbers of columns. SQLite documents `UNION`, `UNION ALL`, `INTERSECT`, and `EXCEPT` as compound `SELECT` operators.

---

# Part XVIII — UNION ALL

## 18. UNION vs UNION ALL

### Question

Return all customer IDs from customers in:

- São Paulo
- Rio de Janeiro

Use `UNION ALL`.

### Answer

```sql
SELECT customer_id
FROM customers
WHERE customer_state = 'SP'

UNION ALL

SELECT customer_id
FROM customers
WHERE customer_state = 'RJ';
```

### Explanation

`UNION ALL` keeps duplicates.

```
UNION
    ↓
combine + remove duplicates

UNION ALL
    ↓
combine + keep duplicates
```

SQLite explicitly distinguishes these behaviors.

---

# Part XIX — INTERSECT

## 19. Customers Who Have Both Orders and Reviews

### Question

Find customer IDs that appear in both:

- `orders`
- `order_reviews`

### Answer

```sql
SELECT customer_id
FROM orders

INTERSECT

SELECT customer_id
FROM order_reviews;
```

### Explanation

`INTERSECT` returns values appearing in both result sets.

Mental model:

```
Orders
   ∩
Reviews
   =
Customers appearing in both
```

---

# Part XX — EXCEPT

## 20. Customers With Orders but No Reviews

### Question

Find customers who have placed an order but do not appear in `order_reviews`.

### Answer

```sql
SELECT customer_id
FROM orders

EXCEPT

SELECT customer_id
FROM order_reviews;
```

### Explanation

Think:

```
Orders
    MINUS
Reviews
```

The result contains IDs found in the first query but not the second.

SQLite defines `EXCEPT` as returning rows from the left query after removing rows appearing in the right query.

---

# Part XXI — Logical Query Processing Order

## 21. Explain the Processing Order

### Question

Consider:

```sql
SELECT
    c.customer_state,
    COUNT(*) AS order_count
FROM customers AS c
JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
HAVING COUNT(*) >= 1000
ORDER BY order_count DESC
LIMIT 5;
```

### Answer

Conceptually reason about it as:

```
FROM
↓
JOIN / ON
↓
WHERE
↓
GROUP BY
↓
HAVING
↓
SELECT
↓
ORDER BY
↓
LIMIT
```

SQLite's documentation presents `SELECT` processing as a series of logical stages and explicitly notes that this is an illustrative model rather than necessarily the physical execution plan used by the engine.

#### 1. FROM

```sql
FROM customers AS c
```

Start with the `customers` table.

#### 2. JOIN / ON

```sql
JOIN orders AS o
    ON c.customer_id = o.customer_id
```

Connect customers to their orders.

#### 3. WHERE

```sql
WHERE o.order_status = 'delivered'
```

Remove individual rows that are not delivered orders.

#### 4. GROUP BY

```sql
GROUP BY c.customer_state
```

Group the remaining orders by customer state.

#### 5. HAVING

```sql
HAVING COUNT(*) >= 1000
```

Remove groups with fewer than 1,000 orders.

#### 6. SELECT

```sql
SELECT
    c.customer_state,
    COUNT(*) AS order_count
```

Produce the requested columns.

#### 7. ORDER BY

```sql
ORDER BY order_count DESC
```

Sort the result from largest order count to smallest.

#### 8. LIMIT

```sql
LIMIT 5
```

Return only the first five rows.

### WHERE vs HAVING

**WHERE**

Filters individual rows:

```sql
WHERE order_status = 'delivered'
```

Think:

> Should this row survive?

**HAVING**

Filters groups:

```sql
HAVING COUNT(*) >= 1000
```

Think:

> Should this group survive?

Therefore:

```
WHERE
↓
rows

GROUP BY
↓
groups

HAVING
↓
groups
```

---

# Part XXII — Customer Order Summary

## 22. Customer Order Summary

### Question

Create a report containing:

- `customer_id`
- `customer_city`
- total number of orders

Include every customer, including customers with zero orders.

Sort by total orders from highest to lowest.

### Answer

```sql
SELECT
    c.customer_id,
    c.customer_city,
    COUNT(o.order_id) AS total_orders
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
GROUP BY
    c.customer_id,
    c.customer_city
ORDER BY total_orders DESC;
```

### Important

Use:

```sql
COUNT(o.order_id)
```

not:

```sql
COUNT(*)
```

**Why?**

A customer with no orders still produces a row because of the `LEFT JOIN`.

For that customer:

```sql
o.order_id = NULL
```

`COUNT(o.order_id)` ignores the `NULL`.

Therefore:

> zero orders → 0

---

# Part XXIII — Category Sales Summary

## 23. Category Sales Summary

### Question

Calculate total sales for each product category.

**Olist Schema Correction**

The Olist `order_items` table does not contain a quantity column.

Each row represents an order item with its own price.

Therefore:

```sql
SUM(oi.price)
```

is the appropriate calculation for the actual Olist dataset.

### Answer

```sql
SELECT
    p.product_category_name,
    SUM(oi.price) AS total_revenue
FROM order_items AS oi
INNER JOIN products AS p
    ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY total_revenue DESC;
```

### Explanation

First:

```sql
INNER JOIN products AS p
    ON oi.product_id = p.product_id
```

connects each order item to its product.

Then:

```sql
GROUP BY p.product_category_name
```

creates one group per category.

Finally:

```sql
SUM(oi.price)
```

calculates total item revenue.

---

# Part XXIV — High-Value Customers

## 24. High-Value Customers

### Question

Find customers whose total order-item spending is greater than 1,000.

### Answer

```sql
SELECT
    o.customer_id,
    SUM(oi.price) AS total_spending
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
GROUP BY o.customer_id
HAVING SUM(oi.price) > 1000
ORDER BY total_spending DESC;
```

### Explanation

First connect:

```
orders
   ↓
order_items
```

using:

```sql
o.order_id = oi.order_id
```

Then group by:

```sql
o.customer_id
```

Then calculate:

```sql
SUM(oi.price)
```

Finally:

```sql
HAVING SUM(oi.price) > 1000
```

keeps only high-value customers.

`HAVING` is required because the condition depends on an aggregate.

---

# Part XXV — Customers Who Bought Above-Average-Priced Products

## 25. Above-Average Product Buyers

### Question

Find customers who purchased at least one product whose price is greater than the average product price.

Return:

- `customer_id`
- `customer_unique_id`

Do not return duplicate customers.

Do not use CTEs or window functions.

### Answer

```sql
SELECT DISTINCT
    c.customer_id,
    c.customer_unique_id
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
INNER JOIN products AS p
    ON oi.product_id = p.product_id
WHERE p.price > (
    SELECT AVG(price)
    FROM products
);
```

### Explanation

The subquery:

```sql
SELECT AVG(price)
FROM products
```

calculates the average product price.

The outer query then checks:

```sql
p.price > average_price
```

The joins follow the actual Olist relationships:

```
customers
    ↓
orders
    ↓
order_items
    ↓
products
```

`DISTINCT` prevents a customer from appearing multiple times if they purchased several above-average products.

---

# Final Integrated Review

## 26. Combined Week 2 Challenge

### Question

Find the top 10 customer states by delivered-order count.

Return:

- `customer_state`
- `order_count`

**Requirements:**

- Join customers to orders.
- Consider only delivered orders.
- Group by customer state.
- Keep states with at least 100 delivered orders.
- Sort from highest to lowest.
- Return only the top 10.

### Answer

```sql
SELECT
    c.customer_state,
    COUNT(*) AS order_count
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
HAVING COUNT(*) >= 100
ORDER BY order_count DESC
LIMIT 10;
```

### Explanation

This combines the major concepts:

```
INNER JOIN
    ↓
connect customers and orders

WHERE
    ↓
keep delivered orders

GROUP BY
    ↓
create one group per state

HAVING
    ↓
keep states with ≥ 100 orders

ORDER BY
    ↓
largest first

LIMIT
    ↓
top 10
```

---

# Conceptual Answer Summary

## INNER JOIN

```sql
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
```

Returns only matching records.

```
A ∩ B
```

## LEFT JOIN

```sql
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
```

Returns everything from the left table.

```
A + matches from B
```

Unmatched B values become `NULL`.

## RIGHT JOIN

```sql
FROM orders AS o
RIGHT JOIN customers AS c
    ON o.customer_id = c.customer_id
```

Same preservation concept as `LEFT JOIN`, but the right table is preserved.

## FULL OUTER JOIN

Returns:

- matches
- unmatched left rows
- unmatched right rows

## NATURAL JOIN

Automatically joins using same-named columns.

Useful to understand.

Usually avoid it in production because changes to table schemas can change which columns participate in the join.

## Set Operations

### UNION

```sql
SELECT ...
UNION
SELECT ...
```

Combines results and removes duplicates.

### UNION ALL

```sql
SELECT ...
UNION ALL
SELECT ...
```

Combines results and keeps duplicates.

### INTERSECT

```sql
SELECT ...
INTERSECT
SELECT ...
```

Returns rows present in both result sets.

### EXCEPT

```sql
SELECT ...
EXCEPT
SELECT ...
```

Returns rows from the first result that are not present in the second.

SQLite requires compound `SELECT`s to have compatible result-column counts, and documents these four operators as its compound-query operators.

## Subqueries

### Non-Correlated

```sql
WHERE price > (
    SELECT AVG(price)
    FROM products
)
```

The inner query does not depend on the outer query.

### Correlated

```sql
WHERE EXISTS (
    SELECT 1
    FROM orders AS o
    WHERE o.customer_id = c.customer_id
)
```

The inner query references the outer query.

### IN

Think:

> Is this value inside this set?

Example:

```sql
WHERE customer_id IN (
    SELECT customer_id
    FROM orders
)
```

### EXISTS

Think:

> Does at least one matching row exist?

Example:

```sql
WHERE EXISTS (
    SELECT 1
    FROM orders AS o
    WHERE o.customer_id = c.customer_id
)
```

## GROUP BY

Creates groups.

```sql
GROUP BY customer_id
```

means:

> one group per customer

Then aggregate functions operate on those groups.

## WHERE

Filters rows.

```sql
WHERE order_status = 'delivered'
```

Think:

> Should this individual row remain?

## HAVING

Filters groups.

```sql
HAVING COUNT(*) > 5
```

Think:

> Should this group remain?

## ORDER BY

Controls the presentation order.

```sql
ORDER BY total_spending DESC
```

means largest first.

## LIMIT

Controls how many rows are returned.

```sql
LIMIT 10
```

means return at most 10 rows.

## Logical Query Processing Order

For the mental model used in this course:

```
FROM
    ↓
JOIN / ON
    ↓
WHERE
    ↓
GROUP BY
    ↓
HAVING
    ↓
SELECT
    ↓
ORDER BY
    ↓
LIMIT
```

Remember that this is a logical reasoning model, not a promise about the database's physical execution plan. SQLite explicitly makes this distinction in its documentation.

---

# Final Week 2 Checklist

Before moving beyond Week 2, I should be able to write and explain:

### Joins

- [ ] INNER JOIN
- [ ] LEFT JOIN
- [ ] RIGHT JOIN
- [ ] FULL OUTER JOIN
- [ ] NATURAL JOIN
- [ ] JOIN ... ON ...
- [ ] NULLs produced by outer joins
- [ ] Difference between LEFT and RIGHT JOIN
- [ ] Risks of NATURAL JOIN

### Filtering and Aggregation

- [ ] WHERE
- [ ] GROUP BY
- [ ] HAVING
- [ ] ORDER BY
- [ ] COUNT()
- [ ] SUM()
- [ ] AVG()
- [ ] WHERE vs HAVING
- [ ] COUNT(*) vs COUNT(column)

### Subqueries

- [ ] Subquery
- [ ] Nested query
- [ ] Non-correlated subquery
- [ ] Correlated subquery
- [ ] IN
- [ ] EXISTS
- [ ] IN vs EXISTS
- [ ] Subquery with AVG()
- [ ] Subquery combined with JOIN

### Set Operations

- [ ] UNION
- [ ] UNION ALL
- [ ] UNION vs UNION ALL
- [ ] INTERSECT
- [ ] EXCEPT

### Query Processing

- [ ] FROM
- [ ] JOIN / ON
- [ ] WHERE
- [ ] GROUP BY
- [ ] HAVING
- [ ] SELECT
- [ ] ORDER BY
- [ ] LIMIT

### Relational Thinking

- [ ] Identify primary keys
- [ ] Identify foreign keys
- [ ] Understand one-to-many relationships
- [ ] Understand how joins multiply rows
- [ ] Understand NULLs
- [ ] Choose INNER vs OUTER JOIN deliberately
- [ ] Choose WHERE vs HAVING deliberately
- [ ] Choose IN vs EXISTS deliberately
- [ ] Recognize when DISTINCT is necessary
- [ ] Combine JOIN + GROUP BY
- [ ] Combine JOIN + HAVING
- [ ] Combine JOIN + subquery
